# **APRENDIZAGEM NÃO SUPERVISIONADA: ASSOCIAÇÃO**

# **ECLAT**

O Eclat (Equivalence Class Clustering and bottom-up Lattice Traversal) é um algoritmo de mineração de regras de associação, assim como o Apriori, mas funciona de maneira mais eficiente em muitos cenários porque usa interseção de conjuntos em vez de múltiplas varreduras no banco de dados.

Como funciona passo a passo:

Transformação dos dados

Os dados de transações (ex.: compras em supermercado) são representados no formato TID-list (Transaction ID list).

Exemplo:

{leite} → {1, 2, 4} (foi comprado nas transações 1, 2 e 4)

{pão} → {2, 3, 4}

{cerveja} → {1, 3}

Cálculo do suporte via interseção

Em vez de percorrer todo o banco de dados, o suporte de um itemset é obtido pela interseção das TID-lists.

Exemplo: suporte de {leite, pão} = interseção de {1,2,4} e {2,3,4} = {2,4} → suporte = 2.

Expansão de itemsets

O algoritmo começa com itemsets de 1 item (singletons).

Em seguida, combina esses itens e calcula o suporte via interseção.

Isso continua de forma recursiva até não haver mais combinações que atendam ao suporte mínimo.

Geração de regras de associação

Depois de encontrar os itemsets frequentes, pode-se derivar regras do tipo
{leite, pão} → {manteiga}
com base em suporte e confiança, assim como no Apriori.

In [16]:
# !pip install pyECLAT

In [17]:
from pyECLAT import Example1

In [18]:
dados = Example1().get()

In [19]:
dados

,0,1,2,3
0,milk,beer,bread,butter
1,coffe,bread,butter,NaN
2,coffe,bread,butter,NaN
3,milk,coffe,bread,butter
4,beer,NaN,NaN,NaN
5,butter,NaN,NaN,NaN
6,bread,NaN,NaN,NaN
7,bean,NaN,NaN,NaN
8,rice,bean,NaN,NaN
9,rice,NaN,NaN,NaN


In [20]:
from pyECLAT import ECLAT
eclat = ECLAT(data=dados, verbose=True)

100%|██████████| 7/7 [00:00<00:00, 3104.26it/s]


In [21]:
eclat.df_bin

,bread,coffe,milk,butter,bean,beer,rice
0,1,0,1,1,0,1,0
1,1,1,0,1,0,0,0
2,1,1,0,1,0,0,0
3,1,1,1,1,0,0,0
4,0,0,0,0,0,1,0
5,0,0,0,1,0,0,0
6,1,0,0,0,0,0,0
7,0,0,0,0,1,0,0
8,0,0,0,0,1,0,1
9,0,0,0,0,0,0,1


In [22]:
eclat.uniq_

['bread', 'coffe', 'milk', 'butter', nan, 'bean', 'beer', 'rice']

In [36]:
indices, suporte = eclat.fit(min_support=0.2, min_combination=2, max_combination=4)
# supporte = propabilidade de ocorrência da combinação

Combination 2 by 2


21it [00:00, 349.76it/s]


Combination 3 by 3


35it [00:00, 632.33it/s]


Combination 4 by 4


35it [00:00, 691.02it/s]


In [24]:
indices

{'bread & coffe': [1, 2, 3],
 'bread & milk': [0, 3],
 'bread & butter': [0, 1, 2, 3],
 'coffe & butter': [1, 2, 3],
 'milk & butter': [0, 3],
 'bread & coffe & butter': [1, 2, 3],
 'bread & milk & butter': [0, 3]}

In [25]:
suporte

{'bread & coffe': 0.3,
 'bread & milk': 0.2,
 'bread & butter': 0.4,
 'coffe & butter': 0.3,
 'milk & butter': 0.2,
 'bread & coffe & butter': 0.3,
 'bread & milk & butter': 0.2}

# **APRIORI**

In [26]:
from pyECLAT import Example1
dados = Example1().get()
dados

,0,1,2,3
0,milk,beer,bread,butter
1,coffe,bread,butter,NaN
2,coffe,bread,butter,NaN
3,milk,coffe,bread,butter
4,beer,NaN,NaN,NaN
5,butter,NaN,NaN,NaN
6,bread,NaN,NaN,NaN
7,bean,NaN,NaN,NaN
8,rice,bean,NaN,NaN
9,rice,NaN,NaN,NaN


In [27]:
dados.shape

(10, 4)

In [28]:
from pyECLAT import ECLAT
eclat = ECLAT(data=dados)
dados2 = eclat.df_bin
dados2

,bread,coffe,milk,butter,bean,beer,rice
0,1,0,1,1,0,1,0
1,1,1,0,1,0,0,0
2,1,1,0,1,0,0,0
3,1,1,1,1,0,0,0
4,0,0,0,0,0,1,0
5,0,0,0,1,0,0,0
6,1,0,0,0,0,0,0
7,0,0,0,0,1,0,0
8,0,0,0,0,1,0,1
9,0,0,0,0,0,0,1


In [29]:
# !pip install mlxtend

In [30]:
from mlxtend.frequent_patterns import apriori, association_rules

In [31]:
# Gerando a associação
associacao = apriori(dados2, min_support=0.05, use_colnames=True)

/home/mau/meuambiente/lib/python3.12/site-packages/mlxtend/frequent_patterns/fpcommon.py:161: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


In [32]:
# Colocando em ordem dos mais frequentes
associacao.sort_values("support", ascending=False).head(15)

,support,itemsets
0,0.5,(bread)
3,0.5,(butter)
9,0.4,"(bread, butter)"
1,0.3,(coffe)
7,0.3,"(bread, coffe)"
12,0.3,"(coffe, butter)"
18,0.3,"(bread, coffe, butter)"
4,0.2,(bean)
5,0.2,(beer)
2,0.2,(milk)


In [33]:
# Criando as regras de associação
regras = association_rules(associacao, metric="confidence")

In [34]:
# Colocando em ordem de suporte, confiança ou grau de associação (Lift)
resultado = regras.sort_values("lift", ascending=False)

Suporte: quão frequente é a regra.

Confiança: quão confiável é a regra (probabilidade condicional).

Lift: quão mais forte é a associação do que o acaso.

Convicção: penaliza regras que erram muito, dá uma visão adicional.

In [35]:
resultado.head(30)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
28,"(butter, beer)","(bread, milk)",0.1,0.2,0.1,1.0,5.0,1.0,0.08,inf,0.888889,0.500000,1.0,0.750
24,"(butter, bread, beer)",(milk),0.1,0.2,0.1,1.0,5.0,1.0,0.08,inf,0.888889,0.500000,1.0,0.750
26,"(bread, beer)","(milk, butter)",0.1,0.2,0.1,1.0,5.0,1.0,0.08,inf,0.888889,0.500000,1.0,0.750
19,"(butter, beer)",(milk),0.1,0.2,0.1,1.0,5.0,1.0,0.08,inf,0.888889,0.500000,1.0,0.750
13,"(bread, beer)",(milk),0.1,0.2,0.1,1.0,5.0,1.0,0.08,inf,0.888889,0.500000,1.0,0.750
9,(coffe),"(bread, butter)",0.3,0.4,0.3,1.0,2.5,1.0,0.18,inf,0.857143,0.750000,1.0,0.875
27,"(milk, beer)","(bread, butter)",0.1,0.4,0.1,1.0,2.5,1.0,0.06,inf,0.666667,0.250000,1.0,0.625
22,"(coffe, milk)","(bread, butter)",0.1,0.4,0.1,1.0,2.5,1.0,0.06,inf,0.666667,0.250000,1.0,0.625
12,(milk),"(bread, butter)",0.2,0.4,0.2,1.0,2.5,1.0,0.12,inf,0.750000,0.500000,1.0,0.750
8,"(coffe, butter)",(bread),0.3,0.5,0.3,1.0,2.0,1.0,0.15,inf,0.714286,0.600000,1.0,0.800
